# Synthetic Title Prompt Lab

Use this notebook for fast prompt/model experiments with individual LLM calls.

- Edit `system_prompt`, `user_prompt_template`, and `model_name`.
- Run one call and inspect raw + parsed outputs.
- Optionally compare several models on the same prompt.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "synthetic_data").exists():
    ROOT = ROOT.parent
if not (ROOT / "synthetic_data").exists():
    raise RuntimeError("Run this notebook from the repo root or from synthetic_data/.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

ROOT

PosixPath('/Users/jiayizhai/CS/fine-tuning-build-project')

In [2]:
import asyncio
import json
import pandas as pd

from synthetic_data.clients import build_client, resolve_model, supported_model_names
from synthetic_data.generate import (
    build_thinking_config,
    load_params,
    load_seed_titles,
    normalize_openai_reasoning_effort,
    parse_response,
    run_pipeline,
    write_outputs,
)
from synthetic_data.models import TitleVariants

In [3]:
params = load_params("params.synthetic_data")
seed_df = load_seed_titles(Path("synthetic_data/data/seed_titles.csv"))

# ----- Experiment config (edit these) -----
model_name = params["model"]
seed_title = seed_df["seed_title"].iloc[0]
num_examples = params["num_examples_per_title"]
temperature = params["temperature"]
reasoning_effort = params["reasoning_effort"]
api_key_env = params["api_key_env"]
base_url_env = params["base_url_env"]

system_prompt = (
    "You generate realistic, varied job-posting titles. Titles should resemble listings "
    "seen on job boards: occasional locations, teams, seniority, bonuses, or abbreviations. "
    "Avoid repeating a single pattern."
)

user_prompt_template = (
    "Seed title: {seed_title}\n"
    "Return exactly {num_examples} distinct 'in-the-wild' job titles as JSON using schema {schema}.\n"
    "Do not include any prose outside the JSON."
)

print("Selected model:", model_name)
print("Available models:", ", ".join(sorted(supported_model_names())))
print("Seed title:", seed_title)
print("Temperature:", temperature)
print("Reasoning effort:", reasoning_effort)
print("API key env:", api_key_env)
print("Base URL env:", base_url_env)

Selected model: gpt-5-mini
Available models: gemini-2.5-flash, gemini-2.5-flash-lite, gemini-3-flash-preview, gemma-3-27b-it, gpt-4.1-mini, gpt-5-mini, gpt-5.2
Seed title: Software Quality Assurance Engineer (SQA Engineer)
Temperature: None
Reasoning effort: minimal
API key env: AZURE_PROJECT_API_KEY
Base URL env: AZURE_OPENAI_ENDPOINT


In [4]:
user_prompt_template

"Seed title: {seed_title}\nReturn exactly {num_examples} distinct 'in-the-wild' job titles as JSON using schema {schema}.\nDo not include any prose outside the JSON."

In [5]:
async def generate_once(
    model_name: str,
    seed_title: str,
    num_examples: int,
    system_prompt: str,
    user_prompt_template: str,
    api_key_env: str | None,
    base_url_env: str | None,
    temperature: float | None,
    reasoning_effort: str,
):
    model_info = resolve_model(model_name)
    client = build_client(
        model_info,
        api_key_env=api_key_env,
        base_url_env=base_url_env,
        async_mode=True,
    )

    schema = json.dumps(TitleVariants.model_json_schema())
    user_prompt = user_prompt_template.format(
        seed_title=seed_title,
        num_examples=num_examples,
        schema=schema,
    )

    if model_info.provider == "openai":
        normalized_effort = normalize_openai_reasoning_effort(model_info.model, reasoning_effort)
        kwargs = {
            "model": model_info.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "response_format": {"type": "json_object"},
        }
        if normalized_effort is not None:
            kwargs["reasoning_effort"] = normalized_effort
        if temperature is not None:
            kwargs["temperature"] = temperature
        completion = await client.chat.completions.create(**kwargs)
        raw_payload = completion.choices[0].message.content or ""
    else:
        config = {"response_mime_type": "application/json"} | build_thinking_config(
            model_info.model,
            reasoning_effort,
        )
        if temperature is not None:
            config["temperature"] = temperature
        response = await asyncio.to_thread(
            client.models.generate_content,
            model=model_info.model,
            contents="\n\n".join([system_prompt, user_prompt]),
            config=config,
        )
        raw_payload = response.text or ""

    parsed = parse_response(seed_title, raw_payload)
    return model_info, user_prompt, raw_payload, parsed

In [6]:
# Single-call experiment
model_info, rendered_user_prompt, raw_payload, parsed = await generate_once(
    model_name=model_name,
    seed_title=seed_title,
    num_examples=num_examples,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    api_key_env=api_key_env,
    base_url_env=base_url_env,
    temperature=temperature,
    reasoning_effort=reasoning_effort,
)

print(f"Provider: {model_info.provider}")
print(f"Canonical model: {model_info.model}")
print(f"Returned titles: {len(parsed.in_the_wild_titles)}")

Provider: openai
Canonical model: gpt-5-mini
Returned titles: 10


In [7]:
print("=== Rendered User Prompt ===")
print(rendered_user_prompt)
print("\n=== Raw Response Payload ===")
print(raw_payload)
print("\n=== Parsed Titles ===")
pd.DataFrame({"generated_title": parsed.in_the_wild_titles})

=== Rendered User Prompt ===
Seed title: Software Quality Assurance Engineer (SQA Engineer)
Return exactly 10 distinct 'in-the-wild' job titles as JSON using schema {"properties": {"seed_title": {"description": "Original clean job title", "title": "Seed Title", "type": "string"}, "in_the_wild_titles": {"description": "Generated noisy job titles", "items": {"type": "string"}, "maxItems": 50, "minItems": 1, "title": "In The Wild Titles", "type": "array"}}, "required": ["seed_title", "in_the_wild_titles"], "title": "TitleVariants", "type": "object"}.
Do not include any prose outside the JSON.

=== Raw Response Payload ===
{
  "seed_title": "Software Quality Assurance Engineer (SQA Engineer)",
  "in_the_wild_titles": [
    "SQA Engineer - Software QA (Remote/UK)",
    "Senior Software Quality Assurance Engineer (Automation)",
    "Quality Assurance Engineer - SQA / Manual & Automated Testing",
    "Software QA Engineer (SQA) - Backend Services, NYC",
    "QA Engineer (SQA) — Performance & 

,generated_title
0,SQA Engineer - Software QA (Remote/UK)
1,Senior Software Quality Assurance Engineer (Au...
2,Quality Assurance Engineer - SQA / Manual & Au...
3,"Software QA Engineer (SQA) - Backend Services,..."
4,QA Engineer (SQA) — Performance & Reliability
5,"Staff SQA Engineer | Test Automation, Java/Python"
6,Software Quality Assurance Analyst (SQA Engine...
7,Embedded SQA Engineer - C/C++ Firmware Testing
8,Software Quality Assurance (SQA) Engineer - Ag...
9,"Junior SQA / QA Engineer - Entry Level, Traini..."


## One title vs all titles

- **One title:** Use the cells above. Set `seed_title` to any row from `seed_df`, e.g. `seed_df["seed_title"].iloc[0]` (first row) or `seed_df["seed_title"].iloc[42]`, then run `generate_once(...)`. The result is a single `TitleVariants` with `in_the_wild_titles`.
- **All titles:** Use the pipeline in `synthetic_data/generate.py`. It loads every row from `seed_titles.csv`, uses cached responses when available, and generates in parallel. You can either run the CLI (below) or run the next cell from this notebook.

In [9]:
# Generate for ALL seed titles (same as: python -m synthetic_data.generate --params params.synthetic_data)
# Uses cache at params["output_responses"]; writes jittered_titles.csv and metrics.
params_all = load_params("params.synthetic_data")
seed_path = Path("synthetic_data/data/seed_titles.csv")
seed_df_all = load_seed_titles(seed_path)

jitter_df, records, metrics = await run_pipeline(params_all, seed_df_all)
write_outputs(jitter_df, metrics, params_all)
print(f"Done. {metrics['num_seed_titles']} seeds -> {metrics['generated_titles']} titles. Saved to {params_all['output_titles']}")
jitter_df.head(15)

RuntimeError: asyncio.run() cannot be called from a running event loop

## Optional: Compare Multiple Models

Uncomment model names below and run this cell to do one call per model with the same prompt.

In [ ]:
compare_models = [
    "gpt-5-mini",
    "gpt-5.2",
    # "gemini-3-flash-preview",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
]

rows = []
for name in compare_models:
    try:
        info, _, raw, parsed_obj = await generate_once(
            model_name=name,
            seed_title=seed_title,
            num_examples=num_examples,
            system_prompt=system_prompt,
            user_prompt_template=user_prompt_template,
            api_key_env=api_key_env,
            base_url_env=base_url_env,
            temperature=temperature,
            reasoning_effort=reasoning_effort,
        )
        rows.append(
            {
                "model": info.model,
                "provider": info.provider,
                "n_titles": len(parsed_obj.in_the_wild_titles),
                "sample_title": parsed_obj.in_the_wild_titles[0] if parsed_obj.in_the_wild_titles else "",
                "status": "ok",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": name,
                "provider": "",
                "n_titles": 0,
                "sample_title": "",
                "status": f"error: {exc}",
            }
        )

pd.DataFrame(rows)

,model,provider,n_titles,sample_title,status
0,gpt-5-mini,openai,10,SQA Engineer - Automation & Manual Testing (Re...,ok
1,gpt-5.2,,0,,error: Error code: 400 - {'error': {'code': 'u...
2,gemini-2.5-flash-lite,,0,,error: 400 INVALID_ARGUMENT. {'error': {'code'...
3,gemini-2.5-flash,,0,,error: 400 INVALID_ARGUMENT. {'error': {'code'...
